# 📄 Demo 01 — Pipeline d'Ingestion FinRAG

Ce notebook démontre le pipeline d'ingestion complet sur les données d'exemple.

**Objectifs :**
- Charger différents types de documents (PDF, CSV, JSON)
- Extraire les tableaux financiers des PDFs
- Appliquer le chunking intelligent
- Analyser les métadonnées extraites

In [ ]:
import sys
from pathlib import Path

# Add project root to path
ROOT_DIR = Path('.').resolve().parent
sys.path.insert(0, str(ROOT_DIR))
print(f'Root: {ROOT_DIR}')

## 1. Chargement des Documents

In [ ]:
from src.ingestion.document_loader import FinancialDocumentLoader

loader = FinancialDocumentLoader()
samples_dir = ROOT_DIR / 'data' / 'samples'

# Load all sample documents
all_docs = loader.load_directory(samples_dir)
print(f'Documents chargés: {len(all_docs)}')
print()
for doc in all_docs[:5]:
    meta = doc.metadata
    print(f'  [{meta.get("document_type","?")}] {meta.get("filename","?")} (p.{meta.get("page_number","?")})' )
    print(f'    Tickers: {meta.get("ticker_symbols", [])}')

## 2. Extraction des Tableaux PDF

In [ ]:
from src.ingestion.table_extractor import FinancialTableExtractor

extractor = FinancialTableExtractor()

# Extract tables from Apple report
apple_pdf = samples_dir / 'apple_annual_report_2023.pdf'
if apple_pdf.exists():
    table_docs = extractor.extract_tables_from_pdf(apple_pdf)
    print(f'Tableaux extraits: {len(table_docs)}')
    if table_docs:
        print('\nPremier tableau:')
        print(table_docs[0].page_content[:500])
else:
    print('PDF non trouvé. Lancez: python data/generate_samples.py')

## 3. Chunking Intelligent

In [ ]:
from src.ingestion.chunker import IntelligentFinancialChunker, ChunkingStrategy
import pandas as pd

chunker = IntelligentFinancialChunker(
    strategy=ChunkingStrategy.HYBRID,
    chunk_size=512,
    chunk_overlap=51,
)

chunks = chunker.chunk_documents(all_docs[:10])  # Demo on first 10 docs
print(f'Chunks créés: {len(chunks)}')

# Analyse des chunks
stats = {
    'semantic': sum(1 for c in chunks if c.metadata.get('chunk_strategy') == 'semantic'),
    'table_aware': sum(1 for c in chunks if c.metadata.get('chunk_strategy') == 'table_aware'),
    'with_table': sum(1 for c in chunks if c.metadata.get('contains_table')),
    'with_numbers': sum(1 for c in chunks if c.metadata.get('contains_numbers')),
}
print('\nStatistiques:')
for k, v in stats.items():
    print(f'  {k}: {v}')

## 4. Analyse des Métadonnées

In [ ]:
import pandas as pd

# Build metadata DataFrame
records = []
for chunk in chunks:
    meta = chunk.metadata
    records.append({
        'filename': meta.get('filename', '?'),
        'doc_type': meta.get('document_type', '?'),
        'strategy': meta.get('chunk_strategy', '?'),
        'has_table': meta.get('contains_table', False),
        'has_numbers': meta.get('contains_numbers', False),
        'time_period': meta.get('time_period', ''),
        'chunk_len': meta.get('chunk_length', len(chunk.page_content)),
    })

df = pd.DataFrame(records)
print('Aperçu des chunks:')
print(df.head(10).to_string())
print()
print('Distribution par type de document:')
print(df['doc_type'].value_counts())

In [ ]:
# Visualisation
try:
    import plotly.express as px
    fig = px.histogram(
        df, x='chunk_len', color='doc_type',
        title='Distribution des tailles de chunks par type de document',
        labels={'chunk_len': 'Longueur du chunk (caractères)', 'count': 'Nombre'},
        template='plotly_dark'
    )
    fig.show()
except ImportError:
    print('plotly non disponible')